# AutoML · M05: AutoGluon

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | AutoML — Pre-Modelado |
| **Módulo** | M05 — AutoGluon |

---

## 🎯 Qué hace

AutoML de Amazon con stacking multi-capa automático: entrena LightGBM,
CatBoost, XGBoost, Random Forest, Extra Trees, Neural Networks y
WeightedEnsemble con bagging y stacking automáticos.
Es el framework más potente del screening — sus resultados son el
baseline más exigente que Fase 5 debe superar.

## 📋 Requisitos

- `data/automl/dataset_final_tfm.parquet` — 33.621 × 25 (generado por f3_m05)
- Entorno: `tfm_abandono`
- Librería: `autogluon`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/results_autogluon.parquet` | Métricas de todos los modelos |
| `data/automl/autogluon_models/` | Modelos guardados en disco |
| `docs/html/fase_automl/m05_autogluon.html` | Informe HTML |

## 🔄 Flujo

```
data/automl/dataset_final_tfm.parquet (24 features + abandono)
    ↓ Split 70/30 estratificado
    ↓ TabularPredictor (best_quality, auto_stack, time_limit=600s)
    ↓ Evaluación en test: predict + predict_proba + sklearn umbral 0.5
    → results_autogluon.parquet + HTML
```

## ➡️ Siguiente

`fautoml_m06_tabpfn.ipynb` — TabPFN (transformer para tabular)


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
import time
import subprocess
from pathlib import Path

warnings.filterwarnings('ignore')

# Verificar AutoGluon
try:
    import autogluon
except ImportError:
    print('⚠️  Instalando autogluon...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'autogluon'])

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from autogluon.tabular import TabularPredictor
import autogluon

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss
)

# Detectar ROOT robusto
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html
from src.html.render import render_pagina

RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_AUTOML, RUTA_FASE_AUTOML_HTML])

fmt           = formato_numero_es
RANDOM_STATE  = 42
FRAMEWORK     = 'autogluon'
NOTEBOOK_NAME = 'fautoml_m05_autogluon.ipynb'
CARPETA_NB    = 'fase_automl'

print(f'✅ AutoGluon {autogluon.core.__version__} listo')
info_entorno()
print('\n✅ Configuración completada')


✓ Directorios verificados: 2
✅ AutoGluon 1.5.0 listo
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyec

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATASET Y VERIFICACIÓN ANTI-LEAKAGE
# ============================================================================

print('\n' + '='*70)
print('CARGANDO DATASET CANÓNICO')
print('='*70)

ruta_dataset = RUTA_AUTOML / 'dataset_final_tfm.parquet'
df = pd.read_parquet(ruta_dataset)

TARGET     = 'abandono'
n_total    = len(df)
n_features = len(df.columns) - 1
n_abandono = int(df[TARGET].sum())
tasa_ab    = n_abandono / n_total

print(f'\n✓ Dataset : {ruta_dataset.name}')
print(f'  Registros : {fmt(n_total)}')
print(f'  Features  : {n_features}')
print(f'  Abandono  : {fmt(n_abandono)} ({tasa_ab*100:.1f}%)')

# Verificación anti-leakage
COLS_LEAKAGE = [
    'egresado', 'egresado_de_hecho', 'curso_ultimo',
    'anos_inactivo', 'pct_titulacion', 'cred_faltantes',
    'progreso_relativo', 'titulacion', 'per_id_ficticio', 'exp_tit_id'
]
leakage_presente = [c for c in COLS_LEAKAGE if c in df.columns]
if leakage_presente:
    raise ValueError(f'LEAKAGE DETECTADO: {leakage_presente}')
print('\n✅ Verificación anti-leakage superada')

# Split 70/30 estratificado
train_df, test_df = train_test_split(
    df,
    test_size    = 0.30,
    random_state = RANDOM_STATE,
    stratify     = df[TARGET]
)
print(f'\nSplit 70/30 estratificado:')
print(f'  Train : {fmt(len(train_df))} ({(train_df[TARGET]==1).mean()*100:.1f}% abandono)')
print(f'  Test  : {fmt(len(test_df))}  ({(test_df[TARGET]==1).mean()*100:.1f}% abandono)')

print(f'\n📋 Features ({n_features}):')
for i, col in enumerate([c for c in df.columns if c != TARGET], 1):
    print(f'  {i:2d}. {col}')



CARGANDO DATASET CANÓNICO

✓ Dataset : dataset_final_tfm.parquet
  Registros : 33.621
  Features  : 24
  Abandono  : 9.833 (29.2%)

✅ Verificación anti-leakage superada

Split 70/30 estratificado:
  Train : 23.534 (29.2% abandono)
  Test  : 10.087  (29.2% abandono)

📋 Features (24):
   1. cred_superados_anio_1er
   2. nota_1er_anio
   3. nota_acceso
   4. nota_selectividad
   5. via_acceso
   6. orden_preferencia
   7. cupo
   8. tasa_abandono_titulacion
   9. rama
  10. cred_repetidos
  11. tasa_repeticion
  12. sexo
  13. edad_entrada
  14. pais_nombre
  15. provincia
  16. universidad_origen
  17. n_anios_beca
  18. anios_sin_beca
  19. situacion_laboral
  20. n_anios_trabajando
  21. max_pagos
  22. indicador_interrupcion
  23. anios_gap
  24. n_anios_sin_notas


In [3]:
# ============================================================================
# CELDA 3: AUTOGLUON — ENTRENAMIENTO Y EVALUACIÓN
# ============================================================================
# preset=best_quality: bagging 8-fold + stacking multi-capa.
# auto_stack=True: AutoGluon decide la arquitectura de stacking óptima.
# Métricas con sklearn a umbral 0.5 — comparables con M01-M04.
# ============================================================================

print('\n' + '='*70)
print('AUTOGLUON — ENTRENAMIENTO')
print('='*70)

ruta_modelos = RUTA_AUTOML / 'autogluon_models'

print(f'\n🚀 Ejecutando AutoGluon (best_quality, max 10 min)...')
t0 = time.time()

predictor = TabularPredictor(
    label       = TARGET,
    path        = str(ruta_modelos),
    eval_metric = 'roc_auc',
    verbosity   = 2
).fit(
    train_data       = train_df,
    time_limit       = 600,
    presets          = 'best_quality',
    auto_stack       = True,
    dynamic_stacking = False,   # Desactivado: OneDrive bloquea borrado de temporales
    ds_args          = {'clean_up_fits': False}  # No borrar temporales
)
t_total = time.time() - t0

# Leaderboard
lb = predictor.leaderboard(test_df, silent=True)
print(f'\n✅ {len(lb)} modelos entrenados en {t_total:.1f}s')
print(f'\nLeaderboard (top 5):')
print(lb.head(5).to_string(index=False))

# Evaluar cada modelo con sklearn a umbral 0.5
print(f'\n🔄 Evaluando modelos en test set (umbral 0.5)...')
y_true      = test_df[TARGET].values
X_test_cols = test_df.drop(columns=[TARGET])
all_results = []

for model_name in lb['model'].values:
    try:
        y_pred = predictor.predict(X_test_cols, model=model_name).values
        y_prob = predictor.predict_proba(X_test_cols, model=model_name)

        # Probabilidad clase positiva (abandono=1)
        y_prob_pos = y_prob[1].values if 1 in y_prob.columns else y_prob.iloc[:, 1].values

        all_results.append({
            'framework'        : FRAMEWORK,
            'model_name'       : model_name,
            'accuracy'         : accuracy_score(y_true, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'precision'        : precision_score(y_true, y_pred, zero_division=0),
            'recall'           : recall_score(y_true, y_pred, zero_division=0),
            'f1'               : f1_score(y_true, y_pred, zero_division=0),
            'auc_roc'          : roc_auc_score(y_true, y_prob_pos),
            'mcc'              : matthews_corrcoef(y_true, y_pred),
            'kappa'            : cohen_kappa_score(y_true, y_pred),
            'log_loss'         : round(log_loss(y_true, y_prob_pos), 4),
            'train_time_sec'   : round(t_total / max(len(lb), 1), 2),
        })
    except Exception as e:
        print(f'  ⚠️ Error con {model_name}: {e}')

df_resultados = (
    pd.DataFrame(all_results)
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)

print(f'\n--- RANKING FINAL (por F1) ---')
cols_show = ['model_name', 'f1', 'auc_roc', 'precision', 'recall', 'mcc']
print(df_resultados[cols_show].head(10).to_string(index=False))


Verbosity: 2 (Standard Logging)



AUTOGLUON — ENTRENAMIENTO

🚀 Ejecutando AutoGluon (best_quality, max 10 min)...


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          22
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       13.97 GB / 31.68 GB (44.1%)
Disk Space Avail:   537.42 GB / 951.65 GB (56.5%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\autogluon_models"
Train Data Rows:    23534
Train Data Columns: 24
Label Column:       abandono
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [np.int64(0), np.int64(1)]
	If 'binary' is not the correc


✅ 10 modelos entrenados en 606.0s

Leaderboard (top 5):
                 model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
   WeightedEnsemble_L2    0.957994   0.956057     roc_auc       10.079982      11.971207 455.456652                 0.037624                0.006275           1.382272            2       True         10
       LightGBM_BAG_L1    0.957108   0.953107     roc_auc        2.100753       0.628863  19.313112                 2.100753                0.628863          19.313112            1       True          2
        XGBoost_BAG_L1    0.956411   0.952718     roc_auc        1.075333       0.548472  29.495607                 1.075333                0.548472          29.495607            1       True          9
       CatBoost_BAG_L1    0.956395   0.952856     roc_auc        0.585868       2.179345 169.993967                 0.585868       

In [4]:
# ============================================================================
# CELDA 4: GUARDAR RESULTADOS
# ============================================================================

ruta_resultados = RUTA_AUTOML / 'results_autogluon.parquet'
df_resultados.to_parquet(ruta_resultados, index=False)
print(f'💾 {ruta_resultados.name}: {len(df_resultados)} modelos guardados')


💾 results_autogluon.parquet: 10 modelos guardados


In [5]:
# ============================================================================
# CELDA 5: GRÁFICOS Y HTML
# ============================================================================

graficos_b64 = {}

# --- Gráfico 1: Top 10 barras horizontales ---
df_plot = df_resultados.head(10).sort_values('f1', ascending=True).copy()
fig, ax = plt.subplots(figsize=(12, 7))
y_pos = np.arange(len(df_plot))
bars  = ax.barh(y_pos, df_plot['f1'], color='#ed8936', alpha=0.85, height=0.6)
ax.scatter(df_plot['auc_roc'], y_pos, color='#e53e3e', s=60, zorder=5, label='AUC-ROC')
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['model_name'], fontsize=9)
ax.set_xlabel('Score')
ax.set_title('AutoGluon: Top 10 Modelos (umbral 0.5)', fontweight='bold', fontsize=14)
ax.axvline(x=0.5, color='gray', ls='--', alpha=0.3)
ax.legend(fontsize=9)
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, df_plot['f1']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8, color='#2d3748')
plt.tight_layout()
graficos_b64['top10'] = figura_a_base64(fig)
plt.close()

# --- Gráfico 2: Top 5 barras agrupadas ---
df_top5   = df_resultados.head(5).copy()
metricas  = ['accuracy', 'f1', 'auc_roc', 'mcc', 'kappa']
etiquetas = ['Exactitud', 'F1', 'AUC-ROC', 'MCC', 'Kappa']
colores   = ['#3182ce', '#38a169', '#e53e3e', '#805ad5', '#ed8936']

fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(df_top5))
width = 0.15
for j, (met, etiq, col) in enumerate(zip(metricas, etiquetas, colores)):
    ax.bar(x + j*width, df_top5[met], width, label=etiq, color=col)
ax.set_xticks(x + width*2)
ax.set_xticklabels(df_top5['model_name'], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('AutoGluon: Top 5 — Métricas Detalladas', fontweight='bold', fontsize=14)
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color='gray', ls='--', alpha=0.3)
plt.tight_layout()
graficos_b64['top5'] = figura_a_base64(fig)
plt.close()

# --- KPIs ---
mejor        = df_resultados.iloc[0]
nombre_mejor = mejor['model_name']
KPIS = [
    {'valor': fmt(n_total),            'titulo': 'Expedientes'},
    {'valor': str(n_features),         'titulo': 'Features'},
    {'valor': str(len(df_resultados)), 'titulo': 'Modelos evaluados'},
    {'valor': f'{mejor["f1"]:.4f}',    'titulo': f'Mejor F1 ({nombre_mejor})'},
]

# --- Sección metodología ---
s_metod = generar_seccion_html('Metodología', f'''
<p><strong>AutoGluon {autogluon.core.__version__}</strong> (Amazon) entrena
{len(df_resultados)} modelos con stacking multi-capa automático sobre
{fmt(n_total)} expedientes y {n_features} features.</p>
<div style="background:#fffaf0;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #ed8936;">
  <strong>Config:</strong> preset=best_quality, time_limit=600s, auto_stack=True,
  eval_metric=roc_auc. Incluye bagging 8-fold y stacking multi-capa.
</div>
<div style="background:#ebf8ff;padding:12px;border-radius:8px;
            margin-top:10px;border-left:4px solid #3182ce;">
  <strong>Evaluación:</strong> Métricas calculadas con sklearn a umbral 0.5 fijo
  sobre test set (30%) — comparables con M01, M02, M03 y M04.
</div>''', 'ℹ️')

# --- Sección gráficos + tabla ---
graf_top10 = f'<img src="data:image/png;base64,{graficos_b64["top10"]}" style="max-width:100%;border-radius:8px;">'
graf_top5  = f'<img src="data:image/png;base64,{graficos_b64["top5"]}" style="max-width:100%;border-radius:8px;">'
s_graficos = generar_seccion_html('Top 10 modelos', graf_top10 + '<br>' + graf_top5, '📊')

tabla = '<table style="width:100%;border-collapse:collapse;font-size:11px;">\n'
tabla += '<tr style="background:#ed8936;color:white;">'
for col in ['#', 'Modelo', 'F1', 'AUC', 'Precision', 'Recall', 'MCC', 'Kappa', 'LogLoss']:
    tabla += f'<th style="padding:8px;text-align:center;">{col}</th>'
tabla += '</tr>\n'
for rank, (i, row) in enumerate(df_resultados.iterrows(), 1):
    bg = '#f7fafc' if rank % 2 == 0 else 'white'
    medallas = {1: '🥇', 2: '🥈', 3: '🥉'}
    rank_str = medallas.get(rank, str(rank))
    nombre_modelo = row['model_name']
    tabla += f'<tr style="background:{bg};">'
    tabla += f'<td style="padding:6px;text-align:center;">{rank_str}</td>'
    tabla += f'<td style="padding:6px;">{nombre_modelo}</td>'
    for campo in ['f1', 'auc_roc', 'precision', 'recall', 'mcc', 'kappa']:
        v = row[campo]
        color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
        tabla += f'<td style="text-align:center;color:{color};">{v:.4f}</td>'
    tabla += f'<td style="text-align:center;">{row["log_loss"]:.4f}</td>'
    tabla += '</tr>\n'
tabla += '</table>'
s_tabla = generar_seccion_html('Ranking completo de modelos', tabla, '🏆')

# --- Comparativa acumulada M01-M05 ---
s_comparativa = ''
frameworks_prev = []
for fname, flabel, tiene_cv in [
    ('results_baselines.parquet',   'M01 Baselines',   True),
    ('results_lazypredict.parquet', 'M02 LazyPredict', False),
    ('results_pycaret.parquet',     'M03 PyCaret',     True),
    ('results_h2o.parquet',         'M04 H2O',         True),
]:
    ruta_fw = RUTA_AUTOML / fname
    if ruta_fw.exists():
        df_fw    = pd.read_parquet(ruta_fw)
        no_dummy = ~df_fw['model_name'].str.startswith('Dummy')
        frameworks_prev.append((flabel, df_fw[no_dummy], tiene_cv))

if frameworks_prev:
    comp = '<table style="width:100%;border-collapse:collapse;">\n'
    comp += '<tr style="background:#ed8936;color:white;">'
    for col in ['Módulo', 'Mejor Modelo', 'F1', 'AUC', 'MCC', 'CV']:
        comp += f'<th style="padding:10px;text-align:center;">{col}</th>'
    comp += '</tr>\n'

    for fw_name, df_fw, cv_real in frameworks_prev:
        mejor_fw  = df_fw.sort_values('f1', ascending=False).iloc[0]
        nombre_fw = mejor_fw['model_name']
        cv_str    = 'CV=5 ✅' if cv_real else 'Sin CV ⚠️'
        comp += f'<tr style="background:#f7fafc;">'
        comp += f'<td style="padding:8px;">{fw_name}</td>'
        comp += f'<td style="padding:8px;">{nombre_fw}</td>'
        comp += f'<td style="text-align:center;">{mejor_fw["f1"]:.4f}</td>'
        comp += f'<td style="text-align:center;">{mejor_fw["auc_roc"]:.4f}</td>'
        comp += f'<td style="text-align:center;">{mejor_fw["mcc"]:.4f}</td>'
        comp += f'<td style="text-align:center;">{cv_str}</td></tr>\n'

    nombre_ag = mejor['model_name']
    comp += f'<tr style="background:white;font-weight:bold;">'
    comp += f'<td style="padding:8px;">M05 AutoGluon</td>'
    comp += f'<td style="padding:8px;">{nombre_ag}</td>'
    comp += f'<td style="text-align:center;">{mejor["f1"]:.4f}</td>'
    comp += f'<td style="text-align:center;">{mejor["auc_roc"]:.4f}</td>'
    comp += f'<td style="text-align:center;">{mejor["mcc"]:.4f}</td>'
    comp += f'<td style="text-align:center;">Bagging+Stack ✅</td></tr>\n'
    comp += '</table>'
    s_comparativa = generar_seccion_html('Comparativa acumulada M01 → M05', comp, '🔄')

# --- Render HTML ---
ruta_html = RUTA_FASE_AUTOML_HTML / 'm05_autogluon.html'
render_pagina(
    NOTEBOOK_NAME,
    generar_kpis_html(KPIS) + s_metod + s_graficos + s_tabla + s_comparativa,
    ruta_html,
    carpeta_notebook=CARPETA_NB
)
print(f'\n✅ HTML: {ruta_html}')



✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m05_autogluon.html


In [6]:
# ============================================================================
# CELDA 6: RESUMEN FINAL
# ============================================================================

mejor = df_resultados.iloc[0]

print('\n' + '='*60)
print('✅ AUTOML-M05 COMPLETADO')
print('='*60)
print(f'Framework     : AutoGluon {autogluon.core.__version__}')
print(f'Dataset       : dataset_final_tfm.parquet ({fmt(n_total)} x {n_features+1})')
print(f'Técnica       : Multi-layer stacking + bagging')
print(f'Modelos       : {len(df_resultados)}')
print(f'Mejor modelo  : {mejor["model_name"]}')
print(f'  F1          : {mejor["f1"]:.4f}')
print(f'  AUC         : {mejor["auc_roc"]:.4f}')
print(f'  MCC         : {mejor["mcc"]:.4f}')
print(f'Modelos disco : {RUTA_AUTOML / "autogluon_models"}')
print(f'Resultados    : {RUTA_AUTOML / "results_autogluon.parquet"}')
print(f'HTML          : {RUTA_FASE_AUTOML_HTML / "m05_autogluon.html"}')
print(f'\n➡️  Siguiente  : fautoml_m06_tabpfn.ipynb')
print('='*60)



✅ AUTOML-M05 COMPLETADO
Framework     : AutoGluon 1.5.0
Dataset       : dataset_final_tfm.parquet (33.621 x 25)
Técnica       : Multi-layer stacking + bagging
Modelos       : 10
Mejor modelo  : WeightedEnsemble_L2
  F1          : 0.8330
  AUC         : 0.9580
  MCC         : 0.7714
Modelos disco : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\autogluon_models
Resultados    : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\results_autogluon.parquet
HTML          : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m05_autogluon.html

➡️  Siguiente  : fautoml_m06_tabpfn.ipynb
